# 04 — Optuna tuning + LightGBM/XGBoost blend

Builds on `03_modeling.ipynb` (LightGBM baseline, 5-fold CV AUC 0.9417).

1. Tune LightGBM and XGBoost with Optuna on a single stratified hold-out split (fast search).
2. Re-train the best configuration of each with the same 5-fold CV as notebook 03 → out-of-fold (OOF) predictions.
3. Pick the blend weight on OOF AUC, apply it to the fold-averaged test predictions → `submission_v2.csv`.

In [1]:
import warnings
import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score

optuna.logging.set_verbosity(optuna.logging.WARNING)
# LightGBM 4.7 deprecates eval_set in favour of eval_X/eval_y; keep the 03-notebook API for consistency
warnings.filterwarnings("ignore", category=lgb.basic.LGBMDeprecationWarning)
SEED = 42
N_TRIALS_LGB = 30
N_TRIALS_XGB = 20

train = pd.read_parquet("../data/train_features.parquet")
test = pd.read_parquet("../data/test_features.parquet")

target = "Will_Buy_EV_bin"
feature_cols = [c for c in train.columns if c != target]
X, y = train[feature_cols], train[target]
X_test = test[feature_cols]

X_tr, X_ho, y_tr, y_ho = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
X.shape, X_tr.shape, X_ho.shape

((668665, 20), (534932, 20), (133733, 20))

## LightGBM search

In [2]:
def lgb_objective(trial):
    params = {
        "n_estimators": 3000,
        "learning_rate": 0.05,
        "num_leaves": trial.suggest_int("num_leaves", 15, 255, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 300, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "subsample_freq": 1,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "random_state": SEED,
        "verbosity": -1,
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_ho, y_ho)], eval_metric="auc",
              callbacks=[lgb.early_stopping(100, verbose=False)])
    trial.set_user_attr("best_iter", model.best_iteration_)
    return roc_auc_score(y_ho, model.predict_proba(X_ho)[:, 1])

lgb_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
lgb_study.optimize(lgb_objective, n_trials=N_TRIALS_LGB)
print(f"Best hold-out AUC: {lgb_study.best_value:.5f}")
lgb_study.best_params

Best hold-out AUC: 0.94179


{'num_leaves': 17,
 'min_child_samples': 189,
 'subsample': 0.8005575058716043,
 'colsample_bytree': 0.8248435466776273,
 'reg_alpha': 0.0012087541473056963,
 'reg_lambda': 7.5794799533480015}

## XGBoost search

In [3]:
def xgb_objective(trial):
    params = {
        "n_estimators": 3000,
        "learning_rate": 0.05,
        "tree_method": "hist",
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 50.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "gamma": trial.suggest_float("gamma", 1e-3, 5.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "eval_metric": "auc",
        "early_stopping_rounds": 100,
        "random_state": SEED,
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_ho, y_ho)], verbose=False)
    trial.set_user_attr("best_iter", model.best_iteration)
    return roc_auc_score(y_ho, model.predict_proba(X_ho)[:, 1])

xgb_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS_XGB)
print(f"Best hold-out AUC: {xgb_study.best_value:.5f}")
xgb_study.best_params

Best hold-out AUC: 0.94177


{'max_depth': 3,
 'min_child_weight': 6.516837053196983,
 'subsample': 0.6634843911688526,
 'colsample_bytree': 0.6146246942909518,
 'gamma': 0.33715297049986,
 'reg_alpha': 0.086082165309894,
 'reg_lambda': 0.1252566644817571}

## 5-fold CV with the tuned parameters (same folds as notebook 03)

In [4]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def run_cv(make_model, fit_kwargs):
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    scores = []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        model = make_model()
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx], eval_set=[(X.iloc[val_idx], y.iloc[val_idx])], **fit_kwargs)
        oof[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
        test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits
        scores.append(roc_auc_score(y.iloc[val_idx], oof[val_idx]))
        print(f"Fold {fold}: AUC={scores[-1]:.5f}")
    print(f"Mean AUC: {np.mean(scores):.5f} (+/- {np.std(scores):.5f}) | OOF AUC: {roc_auc_score(y, oof):.5f}")
    return oof, test_pred

lgb_params = {**lgb_study.best_params, "n_estimators": 3000, "learning_rate": 0.05,
              "subsample_freq": 1, "random_state": SEED, "verbosity": -1}
oof_lgb, test_lgb = run_cv(
    lambda: lgb.LGBMClassifier(**lgb_params),
    {"eval_metric": "auc", "callbacks": [lgb.early_stopping(100, verbose=False)]},
)

Fold 0: AUC=0.94066


Fold 1: AUC=0.94163


Fold 2: AUC=0.94304


Fold 3: AUC=0.94244


Fold 4: AUC=0.94186


Mean AUC: 0.94192 (+/- 0.00080) | OOF AUC: 0.94191


In [5]:
xgb_params = {**xgb_study.best_params, "n_estimators": 3000, "learning_rate": 0.05, "tree_method": "hist",
              "eval_metric": "auc", "early_stopping_rounds": 100, "random_state": SEED}
oof_xgb, test_xgb = run_cv(lambda: xgb.XGBClassifier(**xgb_params), {"verbose": False})

Fold 0: AUC=0.94073


Fold 1: AUC=0.94167


Fold 2: AUC=0.94309


Fold 3: AUC=0.94249


Fold 4: AUC=0.94200


Mean AUC: 0.94199 (+/- 0.00079) | OOF AUC: 0.94199


## Blend

AUC only depends on ranking, so blend rank-normalised predictions and choose the LightGBM weight on OOF AUC.

In [6]:
def to_rank(p):
    return pd.Series(p).rank(pct=True).to_numpy()

weights = np.linspace(0, 1, 21)
blend_scores = [roc_auc_score(y, w * to_rank(oof_lgb) + (1 - w) * to_rank(oof_xgb)) for w in weights]
best_w = weights[int(np.argmax(blend_scores))]

summary = pd.DataFrame({
    "model": ["LightGBM baseline (nb 03)", "LightGBM tuned", "XGBoost tuned", f"Blend (w_lgb={best_w:.2f})"],
    "oof_auc": [0.9417, roc_auc_score(y, oof_lgb), roc_auc_score(y, oof_xgb), max(blend_scores)],
})
summary

,model,oof_auc
0,LightGBM baseline (nb 03),0.941700
1,LightGBM tuned,0.941915
2,XGBoost tuned,0.941987
3,Blend (w_lgb=0.45),0.942072


In [7]:
final_preds = best_w * to_rank(test_lgb) + (1 - best_w) * to_rank(test_xgb)
submission = pd.DataFrame({"id": test["id"], "Will_Buy_EV": final_preds})
submission.to_csv("../submissions/submission_v2.csv", index=False)
submission.shape, submission.head()

((286571, 2),
        id  Will_Buy_EV
 0  668665     0.429390
 1  668666     0.507814
 2  668667     0.334482
 3  668668     0.246191
 4  668669     0.511768)